**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 13: 데이터 파이프라인 & 합성 데이터 / Distillation

---

### 📋 학습 목표

본 세션은 Session 11(파인튜닝 개념 정리)에서 다룬 SFT/IT 학습 데이터를 **실제로 준비하는 방법**을 다룹니다. 두 가지 흐름으로 나뉩니다.

**Part A — 데이터 파이프라인 (수집·정제·증강·변환)**
- 🔹 Alpaca / ShareGPT / ChatML 등 SFT 데이터 형식의 차이를 이해합니다
- 🔹 데이터 수집 → 정제(cleaning) → 증강(augmentation) 파이프라인을 구현합니다
- 🔹 HuggingFace `datasets` 라이브러리로 데이터 로드/저장/공유합니다
- 🔹 데이터 품질 검증 체크리스트를 적용합니다

**Part B — 합성 데이터 & Distillation**
- 🔹 합성 데이터(synthetic data)가 필요한 이유와 Self-Instruct 방법론을 이해합니다
- 🔹 OpenAI API(GPT-4 / GPT-4o-mini)로 seed 데이터를 확장합니다
- 🔹 Distillation: 큰 모델의 출력으로 작은 모델 학습용 데이터를 만듭니다
- 🔹 API 비용을 계산하고 최적화 팁을 적용합니다

### 📦 필요 라이브러리

```
openai, python-dotenv, datasets, numpy
```

### ⏱️ 예상 소요 시간: 약 120분

### ⚠️ 주의

Part B의 데이터 생성 셀들은 **OpenAI 크레딧을 소모**합니다 (gpt-4o-mini 기준 노트북 전체 ≈ $1 미만).
API 키는 `.env`의 `OPENAI_API_KEY`에서 자동 로드합니다.

---

In [1]:
# 환경 설정 (Part A + Part B 공용)
import os
import json
import random
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# 데이터 경로 — 실행 위치(part2/ 또는 레포 루트)와 무관하게 data/samples 를 탐색
def _find_data_dir():
    here = os.path.abspath(os.getcwd())
    for _ in range(5):
        cand = os.path.join(here, "data", "samples")
        if os.path.isdir(cand):
            return cand
        here = os.path.dirname(here)
    fallback = os.path.join(os.path.dirname(os.getcwd()), "data", "samples")
    os.makedirs(fallback, exist_ok=True)
    return fallback

DATA_DIR = _find_data_dir()
SEED_FILE = os.path.join(DATA_DIR, "alpaca_ko_sample.json")
print(f"\U0001F4C1 DATA_DIR = {DATA_DIR}")

# OpenAI 클라이언트 (Part B에서 사용)
try:
    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key and not api_key.startswith("sk-your"):
        client = OpenAI(api_key=api_key)
        print(f"\u2705 OpenAI 클라이언트 준비 완료 (key: {api_key[:10]}...)")
    else:
        client = None
        print("\u26A0\uFE0F OPENAI_API_KEY가 설정되지 않았습니다. Part B 셀은 건너뜁니다.")
except ImportError:
    client = None
    print("\u26A0\uFE0F openai 패키지 미설치. pip install openai")

random.seed(42)
np.random.seed(42)
print("\u2705 환경 설정 완료")


📁 DATA_DIR = /home/hpe/LLM_master_5parts/data/samples
✅ OpenAI 클라이언트 준비 완료 (key: sk-proj-Mq...)
✅ 환경 설정 완료


---

# 📦 Part A — 데이터 파이프라인

이 영역은 SFT/Instruction Tuning에 사용할 데이터를 **수집·정제·증강·변환**하는 전체 흐름을 다룹니다.

---
## 🎯 1. 파인튜닝 데이터 파이프라인 개요

### 전체 파이프라인

```
📥 데이터 수집 (Collection)
    ↓
🔍 데이터 정제 (Cleaning)
    ↓
📐 데이터 형식 변환 (Formatting)
    ↓
📊 데이터 증강 (Augmentation) [선택]
    ↓
✅ 품질 검증 (Validation)
    ↓
🚀 학습 데이터 준비 완료
```

### 데이터 품질이 중요한 이유

- 📌 **"Garbage In, Garbage Out"** - 나쁜 데이터로는 좋은 모델을 만들 수 없음
- 📌 LIMA 논문 (2023): **1,000개의 고품질 데이터**로 GPT-4 수준의 응답 가능
- 📌 데이터 양보다 **데이터 품질**이 더 중요
- 📌 현업에서 파인튜닝 시간의 70%는 데이터 준비에 사용

---
## 1️⃣ 2. 데이터 형식: Alpaca vs ShareGPT vs ChatML

### Alpaca 형식
Stanford Alpaca 프로젝트에서 사용한 형식으로, 가장 단순합니다.

```json
{
    "instruction": "지시 내용",
    "input": "입력 데이터 (선택)",
    "output": "기대 출력"
}
```

### ShareGPT 형식
멀티턴 대화를 지원하는 형식입니다.

```json
{
    "conversations": [
        {"from": "human", "value": "사용자 메시지"},
        {"from": "gpt", "value": "AI 응답"},
        {"from": "human", "value": "후속 질문"},
        {"from": "gpt", "value": "후속 응답"}
    ]
}
```

### ChatML 형식
OpenAI의 Chat Completion API와 호환되는 형식입니다.

```json
{
    "messages": [
        {"role": "system", "content": "시스템 프롬프트"},
        {"role": "user", "content": "사용자 메시지"},
        {"role": "assistant", "content": "AI 응답"}
    ]
}
```

In [2]:
# 각 형식 예시 생성 및 비교
print("📊 데이터 형식 비교")
print("=" * 60)

# 1. Alpaca 형식
alpaca_example = {
    "instruction": "다음 문장의 감성을 분석하세요.",
    "input": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다.",
    "output": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
}

print("\n📋 1. Alpaca 형식:")
print(json.dumps(alpaca_example, ensure_ascii=False, indent=2))

# 2. ShareGPT 형식
sharegpt_example = {
    "conversations": [
        {"from": "human", "value": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."},
        {"from": "gpt", "value": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"},
        {"from": "human", "value": "역순으로 정렬하려면요?"},
        {"from": "gpt", "value": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"}
    ]
}

print("\n📋 2. ShareGPT 형식:")
print(json.dumps(sharegpt_example, ensure_ascii=False, indent=2))

# 3. ChatML 형식
chatml_example = {
    "messages": [
        {"role": "system", "content": "당신은 한국어 감성 분석 전문가입니다."},
        {"role": "user", "content": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."},
        {"role": "assistant", "content": "감성 분석 결과:\n- 음식: 긍정 (맛있었지만)\n- 서비스: 부정 (너무 느렸습니다)\n- 종합: 혼합 (약간 긍정)"}
    ]
}

print("\n📋 3. ChatML 형식:")
print(json.dumps(chatml_example, ensure_ascii=False, indent=2))

📊 데이터 형식 비교

📋 1. Alpaca 형식:
{
  "instruction": "다음 문장의 감성을 분석하세요.",
  "input": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다.",
  "output": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
}

📋 2. ShareGPT 형식:
{
  "conversations": [
    {
      "from": "human",
      "value": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."
    },
    {
      "from": "gpt",
      "value": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"
    },
    {
      "from": "human",
      "value": "역순으로 정렬하려면요?"
    },
    {
      "from": "gpt",
      "value": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"
    }
  ]
}

📋 3. ChatML 형식:
{
  "messages": [
    {
      "role": "system",
      "content": "당신은 한국어 감성 분석 전문가입니다."
    },
    {
      "role": "user",
      "content": "이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."
    },
    {
      "role": "assist

In [3]:
# 형식 간 변환 함수
print("🔄 데이터 형식 변환 함수")
print("=" * 60)

def alpaca_to_chatml(alpaca_data):
    """Alpaca → ChatML 변환"""
    messages = []
    
    # user 메시지 구성
    user_content = alpaca_data["instruction"]
    if alpaca_data.get("input", "").strip():
        user_content += f"\n\n{alpaca_data['input']}"
    
    messages.append({"role": "user", "content": user_content})
    messages.append({"role": "assistant", "content": alpaca_data["output"]})
    
    return {"messages": messages}

def sharegpt_to_chatml(sharegpt_data):
    """ShareGPT → ChatML 변환"""
    role_map = {"human": "user", "gpt": "assistant", "system": "system"}
    messages = []
    
    for conv in sharegpt_data["conversations"]:
        role = role_map.get(conv["from"], conv["from"])
        messages.append({"role": role, "content": conv["value"]})
    
    return {"messages": messages}

def chatml_to_alpaca(chatml_data):
    """ChatML → Alpaca 변환 (단일 턴만)"""
    messages = chatml_data["messages"]
    user_msg = ""
    assistant_msg = ""
    
    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg["content"]
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"]
    
    return {
        "instruction": user_msg,
        "input": "",
        "output": assistant_msg
    }

# 변환 테스트
print("\n🔄 Alpaca → ChatML 변환:")
converted = alpaca_to_chatml(alpaca_example)
print(json.dumps(converted, ensure_ascii=False, indent=2))

print("\n🔄 ShareGPT → ChatML 변환:")
converted2 = sharegpt_to_chatml(sharegpt_example)
print(json.dumps(converted2, ensure_ascii=False, indent=2))

print("\n💡 실무에서는 ChatML 형식으로 통일하는 것을 권장합니다.")
print("   (대부분의 최신 모델이 ChatML 형식을 지원)")

🔄 데이터 형식 변환 함수

🔄 Alpaca → ChatML 변환:
{
  "messages": [
    {
      "role": "user",
      "content": "다음 문장의 감성을 분석하세요.\n\n이 식당의 음식은 정말 맛있었지만, 서비스가 너무 느렸습니다."
    },
    {
      "role": "assistant",
      "content": "이 문장은 혼합된 감성을 보여줍니다. 음식의 맛에 대해서는 긍정적('정말 맛있었지만')이나, 서비스 속도에 대해서는 부정적('너무 느렸습니다')입니다. 전반적으로 중립에서 약간 긍정적인 감성으로 판단됩니다."
    }
  ]
}

🔄 ShareGPT → ChatML 변환:
{
  "messages": [
    {
      "role": "user",
      "content": "파이썬에서 리스트를 정렬하는 방법을 알려주세요."
    },
    {
      "role": "assistant",
      "content": "파이썬에서 리스트를 정렬하는 방법은 크게 두 가지입니다:\n\n1. `sort()` 메서드: 원본 리스트를 직접 정렬\n2. `sorted()` 함수: 새로운 정렬된 리스트 반환"
    },
    {
      "role": "user",
      "content": "역순으로 정렬하려면요?"
    },
    {
      "role": "assistant",
      "content": "`reverse=True` 파라미터를 사용하면 됩니다:\n```python\nmy_list.sort(reverse=True)\n# 또는\nsorted_list = sorted(my_list, reverse=True)\n```"
    }
  ]
}

💡 실무에서는 ChatML 형식으로 통일하는 것을 권장합니다.
   (대부분의 최신 모델이 ChatML 형식을 지원)


---
## 2️⃣ 3. 데이터 수집 방법

### 데이터 소스 분류

| 소스 | 장점 | 단점 | 예시 |
|------|------|------|------|
| **공개 데이터셋** | 무료, 검증됨 | 도메인 제한 | HuggingFace Hub, KorQuAD |
| **합성 데이터** | 대량 생성 가능 | 품질 검증 필요 | GPT-4로 생성 |
| **크롤링** | 도메인 특화 | 저작권 문제 | 뉴스, 블로그 |
| **직접 제작** | 최고 품질 | 비용/시간 | 전문가 작성 |

### 한국어 공개 데이터셋

| 데이터셋 | 크기 | 용도 |
|---------|------|------|
| **KoAlpaca** | ~52K | 범용 Instruction |
| **KOpen-platypus** | ~25K | 논리/수학 |
| **KorQuAD 2.0** | ~100K | 질의응답 |
| **NSMC** | ~200K | 감성 분석 |
| **AI Hub** | 다양 | 번역, 요약, QA 등 |

In [4]:
# Seed 데이터: "이렇게 만든다" 형식 예시 + 이미 있는 파일 사용
print("\U0001F4DD Alpaca 형식 Seed 데이터")
print("=" * 60)

# 1) Alpaca 형식이 어떤 구조인지 보여주는 예시 (참고용 — 파일을 덮어쓰지 않음)
sample_example = [
    {
        "instruction": "다음 문장을 영어로 번역하세요.",
        "input": "오늘 날씨가 매우 좋습니다.",
        "output": "The weather is very nice today.",
    },
    {
        "instruction": "머신러닝에서 과적합(Overfitting)이 무엇인지 설명하세요.",
        "input": "",
        "output": "과적합은 모델이 학습 데이터에 지나치게 맞춰져 새 데이터 성능이 떨어지는 현상입니다.",
    },
]
print("\U0001F4CC Alpaca 형식 예시 (instruction / input / output):")
print(json.dumps(sample_example, ensure_ascii=False, indent=2))

# 2) Seed 파일은 이미 저장소에 있으므로 그대로 사용한다.
#    (없을 때만 위 예시로 최소 생성 — 기존 파일은 절대 덮어쓰지 않음)
if not os.path.exists(SEED_FILE):
    with open(SEED_FILE, "w", encoding="utf-8") as f:
        json.dump(sample_example, f, ensure_ascii=False, indent=2)
    print(f"\n\u2139\uFE0F Seed 파일이 없어 예시로 생성했습니다: {SEED_FILE}")

# 3) 실제 데이터셋 로드 — 이후 정제/증강은 이 변수(dataset)를 사용
with open(SEED_FILE, encoding="utf-8") as f:
    dataset = json.load(f)

print(f"\n\u2705 Seed 데이터 로드: {len(dataset)}개  \u2190  {SEED_FILE}")
print("   형식: Alpaca (instruction, input, output)")
print("\n\U0001F4CB 예시 1개:")
print(f"   instruction: {dataset[0]['instruction'][:50]}")
print(f"   output:      {dataset[0]['output'][:50]}...")


📝 Alpaca 형식 Seed 데이터
📌 Alpaca 형식 예시 (instruction / input / output):
[
  {
    "instruction": "다음 문장을 영어로 번역하세요.",
    "input": "오늘 날씨가 매우 좋습니다.",
    "output": "The weather is very nice today."
  },
  {
    "instruction": "머신러닝에서 과적합(Overfitting)이 무엇인지 설명하세요.",
    "input": "",
    "output": "과적합은 모델이 학습 데이터에 지나치게 맞춰져 새 데이터 성능이 떨어지는 현상입니다."
  }
]

✅ Seed 데이터 로드: 50개  ←  /home/hpe/LLM_master_5parts/data/samples/alpaca_ko_sample.json
   형식: Alpaca (instruction, input, output)

📋 예시 1개:
   instruction: LoRA 파인튜닝이 무엇인지 설명하세요.
   output:      LoRA(Low-Rank Adaptation)는 대규모 언어 모델을 효율적으로 파인튜닝하기...


---
## 3️⃣ 4. 데이터 정제 (Cleaning)

### 정제가 필요한 이유
- 📌 중복 데이터 → 학습 편향
- 📌 저품질 데이터 → 모델 성능 저하
- 📌 너무 짧거나 긴 데이터 → 학습 비효율
- 📌 잘못된 형식 → 학습 오류

### 주요 정제 단계

| 단계 | 설명 | 기준 |
|------|------|------|
| 형식 검증 | 필수 필드 존재 여부 | instruction, output 필수 |
| 중복 제거 | 동일/유사 데이터 제거 | 정확 매칭 + 유사도 |
| 길이 필터링 | 너무 짧거나 긴 데이터 제거 | output 10~2000자 |
| 품질 필터링 | 의미 없는 응답 제거 | 빈 응답, 반복 등 |
| 언어 필터링 | 대상 언어가 아닌 데이터 제거 | 한국어 비율 체크 |

In [5]:
# 데이터 정제 파이프라인 구현
print("🔧 데이터 정제 파이프라인 구현")
print("=" * 60)

class DataCleaner:
    """파인튜닝 데이터 정제기"""
    
    def __init__(self, min_output_len=10, max_output_len=2000,
                 min_instruction_len=5):
        self.min_output_len = min_output_len
        self.max_output_len = max_output_len
        self.min_instruction_len = min_instruction_len
        self.stats = {
            "total": 0,
            "passed": 0,
            "removed_format": 0,
            "removed_length": 0,
            "removed_quality": 0,
            "removed_duplicate": 0,
        }
    
    def validate_format(self, item):
        """형식 검증"""
        if not isinstance(item, dict):
            return False
        if "instruction" not in item or "output" not in item:
            return False
        if not item["instruction"].strip() or not item["output"].strip():
            return False
        return True
    
    def check_length(self, item):
        """길이 필터링"""
        output_len = len(item["output"])
        instruction_len = len(item["instruction"])
        
        if output_len < self.min_output_len:
            return False
        if output_len > self.max_output_len:
            return False
        if instruction_len < self.min_instruction_len:
            return False
        return True
    
    def check_quality(self, item):
        """품질 필터링"""
        output = item["output"]
        
        # 반복 텍스트 체크
        words = output.split()
        if len(words) > 5:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio < 0.3:  # 단어의 70% 이상이 반복
                return False
        
        # 의미없는 응답 체크
        low_quality_patterns = [
            "모르겠습니다", "답변할 수 없습니다", "정보가 없습니다",
            "I don't know", "I cannot", "N/A"
        ]
        for pattern in low_quality_patterns:
            if output.strip() == pattern:
                return False
        
        return True
    
    def clean(self, data):
        """전체 정제 파이프라인 실행"""
        self.stats["total"] = len(data)
        cleaned = []
        seen = set()  # 중복 체크용
        
        for item in data:
            # 1. 형식 검증
            if not self.validate_format(item):
                self.stats["removed_format"] += 1
                continue
            
            # 2. 중복 제거
            key = item["instruction"] + item["output"]
            if key in seen:
                self.stats["removed_duplicate"] += 1
                continue
            seen.add(key)
            
            # 3. 길이 필터링
            if not self.check_length(item):
                self.stats["removed_length"] += 1
                continue
            
            # 4. 품질 필터링
            if not self.check_quality(item):
                self.stats["removed_quality"] += 1
                continue
            
            cleaned.append(item)
        
        self.stats["passed"] = len(cleaned)
        return cleaned
    
    def print_report(self):
        """정제 결과 리포트 출력"""
        print(f"\n📊 데이터 정제 결과 리포트")
        print(f"{'='*50}")
        print(f"  총 데이터:     {self.stats['total']:>6d}")
        print(f"  형식 오류:     {self.stats['removed_format']:>6d}")
        print(f"  중복 제거:     {self.stats['removed_duplicate']:>6d}")
        print(f"  길이 필터링:   {self.stats['removed_length']:>6d}")
        print(f"  품질 필터링:   {self.stats['removed_quality']:>6d}")
        print(f"  {'─'*40}")
        print(f"  최종 데이터:   {self.stats['passed']:>6d} ({self.stats['passed']/self.stats['total']*100:.1f}%)")


# 정제 대상: cell 9 에서 로드한 실제 Seed 파일(dataset) + 시연용 불량 데이터 주입
test_data = dataset.copy()
test_data.extend([
    # 중복 데이터
    dataset[0],
    # 형식 오류
    {"instruction": "질문"},  # output 없음
    {"instruction": "", "input": "", "output": "답변"},  # 빈 instruction
    # 너무 짧은 응답
    {"instruction": "이것은 무엇인가요?", "input": "", "output": "네"},
    # 저품질 응답
    {"instruction": "질문입니다", "input": "", "output": "모르겠습니다"},
])

print(f"📌 테스트 데이터: {len(test_data)}개 (정상 {len(dataset)}개 + 불량 {len(test_data)-len(dataset)}개)")

# 정제 실행
cleaner = DataCleaner()
cleaned_data = cleaner.clean(test_data)
cleaner.print_report()

print(f"\n✅ 정제 파이프라인 동작 확인 완료!")

🔧 데이터 정제 파이프라인 구현
📌 테스트 데이터: 55개 (정상 50개 + 불량 5개)

📊 데이터 정제 결과 리포트
  총 데이터:         55
  형식 오류:          2
  중복 제거:          1
  길이 필터링:        2
  품질 필터링:        0
  ────────────────────────────────────────
  최종 데이터:       50 (90.9%)

✅ 정제 파이프라인 동작 확인 완료!


---
## 4️⃣ 5. 데이터 증강 기법

### 주요 증강 기법

| 기법 | 설명 | 적합한 경우 |
|------|------|----------|
| **Paraphrasing** | 같은 의미의 다른 표현 | 지시문 다양화 |
| **Back-Translation** | 번역 후 재번역 | 언어 다양성 |
| **Self-Instruct** | LLM으로 데이터 생성 | 데이터 부족 시 |
| **Distillation** | 큰 모델로 데이터 생성 | 품질 향상 |
| **Template Variation** | 템플릿 변경 | 형식 다양화 |

In [6]:
# 템플릿 기반 데이터 증강
print("🔧 템플릿 기반 데이터 증강 (Instruction Variation)")
print("=" * 60)

# 지시문 템플릿 변형
instruction_templates = {
    "번역": [
        "다음 문장을 영어로 번역하세요.",
        "아래 한국어를 영어로 바꿔주세요.",
        "다음을 영어로 옮겨주세요.",
        "영어 번역을 부탁드립니다.",
        "한영 번역을 해주세요.",
    ],
    "감성분석": [
        "다음 텍스트의 감성을 분석하세요.",
        "아래 글의 감정을 판단해주세요.",
        "이 문장이 긍정적인지 부정적인지 분석하세요.",
        "감성 분석을 수행해주세요.",
        "다음 텍스트의 감정 톤을 파악해주세요.",
    ],
    "요약": [
        "다음 문장을 3줄로 요약하세요.",
        "아래 내용을 간단히 요약해주세요.",
        "핵심 내용만 추려서 정리해주세요.",
        "다음 글의 요점을 정리하세요.",
        "간략한 요약을 작성해주세요.",
    ]
}

def augment_with_templates(data, templates_dict):
    """템플릿을 활용한 데이터 증강"""
    augmented = []
    
    for item in data:
        augmented.append(item)  # 원본 유지
        
        # 매칭되는 카테고리 찾기
        for category, templates in templates_dict.items():
            if any(kw in item["instruction"] for kw in ["번역", "영어로"]):
                if category == "번역":
                    for template in templates:
                        if template != item["instruction"]:
                            new_item = item.copy()
                            new_item["instruction"] = template
                            augmented.append(new_item)
                    break
            elif any(kw in item["instruction"] for kw in ["감성", "감정"]):
                if category == "감성분석":
                    for template in templates:
                        if template != item["instruction"]:
                            new_item = item.copy()
                            new_item["instruction"] = template
                            augmented.append(new_item)
                    break
    
    return augmented

augmented_data = augment_with_templates(dataset, instruction_templates)
print(f"📌 원본 데이터: {len(dataset)}개")
print(f"📌 증강 후 데이터: {len(augmented_data)}개")
print(f"📌 증가율: {(len(augmented_data)/len(dataset)-1)*100:.0f}%")

# 증강된 예시 표시
print(f"\n📋 증강된 번역 데이터 예시:")
for item in augmented_data:
    if "번역" in item["instruction"] or "영어로" in item["instruction"] or "한영" in item["instruction"]:
        print(f"  🔹 {item['instruction']}")

print("\n💡 실제 파인튜닝에서는 GPT-4 등을 활용하여 더 다양하고 자연스러운 증강이 가능합니다.")

🔧 템플릿 기반 데이터 증강 (Instruction Variation)
📌 원본 데이터: 50개
📌 증강 후 데이터: 70개
📌 증가율: 40%

📋 증강된 번역 데이터 예시:
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.
  🔹 다음 문장을 영어로 번역하세요.
  🔹 아래 한국어를 영어로 바꿔주세요.
  🔹 다음을 영어로 옮겨주세요.
  🔹 영어 번역을 부탁드립니다.
  🔹 한영 번역을 해주세요.

💡 실제 파인튜닝에서는 GPT-4 등을 활용하여 더 다양하고 자연스러운 증강이 가능합니다.


---
## 5️⃣ 6. 실습: 샘플 데이터 로드 및 변환

앞에서 생성한 `alpaca_ko_sample.json` 파일을 로드하고 다양한 형식으로 변환해봅시다.

In [7]:
import numpy as np
# 데이터 로드 및 탐색
print("📥 샘플 데이터 로드 및 탐색")
print("=" * 60)

# 파일 로드
with open(os.path.join(DATA_DIR, "alpaca_ko_sample.json"), 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ 로드 완료: {len(data)}개 데이터\n")

# 데이터 통계
instruction_lens = [len(d["instruction"]) for d in data]
input_lens = [len(d.get("input", "")) for d in data]
output_lens = [len(d["output"]) for d in data]

print("📊 데이터 통계:")
print(f"  Instruction 길이: 평균 {np.mean(instruction_lens):.0f}자, "
      f"최소 {min(instruction_lens)}자, 최대 {max(instruction_lens)}자")
print(f"  Input 길이:       평균 {np.mean(input_lens):.0f}자, "
      f"최소 {min(input_lens)}자, 최대 {max(input_lens)}자")
print(f"  Output 길이:      평균 {np.mean(output_lens):.0f}자, "
      f"최소 {min(output_lens)}자, 최대 {max(output_lens)}자")

# Input이 있는/없는 데이터 비율
has_input = sum(1 for d in data if d.get("input", "").strip())
print(f"\n📊 Input 유무:")
print(f"  Input 있음: {has_input}개 ({has_input/len(data)*100:.0f}%)")
print(f"  Input 없음: {len(data)-has_input}개 ({(len(data)-has_input)/len(data)*100:.0f}%)")

📥 샘플 데이터 로드 및 탐색
✅ 로드 완료: 50개 데이터

📊 데이터 통계:
  Instruction 길이: 평균 30자, 최소 17자, 최대 49자
  Input 길이:       평균 3자, 최소 0자, 최대 39자
  Output 길이:      평균 223자, 최소 33자, 최대 304자

📊 Input 유무:
  Input 있음: 5개 (10%)
  Input 없음: 45개 (90%)


In [8]:
# Alpaca → ChatML 전체 변환
print("🔄 Alpaca → ChatML 전체 변환")
print("=" * 60)

def convert_alpaca_to_chatml(alpaca_list, system_prompt=None):
    """Alpaca 데이터 리스트를 ChatML 형식으로 변환"""
    chatml_list = []
    
    for item in alpaca_list:
        messages = []
        
        # System prompt 추가 (선택)
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        # User 메시지
        user_content = item["instruction"]
        if item.get("input", "").strip():
            user_content += f"\n\n{item['input']}"
        messages.append({"role": "user", "content": user_content})
        
        # Assistant 응답
        messages.append({"role": "assistant", "content": item["output"]})
        
        chatml_list.append({"messages": messages})
    
    return chatml_list

# 변환 실행
system_prompt = "당신은 유용하고 정확한 답변을 제공하는 한국어 AI 어시스턴트입니다."
chatml_data = convert_alpaca_to_chatml(data, system_prompt=system_prompt)

# 결과 확인
print(f"✅ 변환 완료: {len(chatml_data)}개\n")

print("📋 변환 결과 예시 (첫 번째 데이터):")
print(json.dumps(chatml_data[0], ensure_ascii=False, indent=2))

# ChatML 형식으로 저장
chatml_file = os.path.join(DATA_DIR, "chatml_ko_sample.json")
with open(chatml_file, 'w', encoding='utf-8') as f:
    json.dump(chatml_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ ChatML 형식으로 저장: {chatml_file}")

🔄 Alpaca → ChatML 전체 변환
✅ 변환 완료: 50개

📋 변환 결과 예시 (첫 번째 데이터):
{
  "messages": [
    {
      "role": "system",
      "content": "당신은 유용하고 정확한 답변을 제공하는 한국어 AI 어시스턴트입니다."
    },
    {
      "role": "user",
      "content": "LoRA 파인튜닝이 무엇인지 설명하세요."
    },
    {
      "role": "assistant",
      "content": "LoRA(Low-Rank Adaptation)는 대규모 언어 모델을 효율적으로 파인튜닝하기 위한 기법입니다. 기존 모델의 가중치를 동결(freeze)하고, 각 레이어에 작은 저랭크(low-rank) 행렬 A와 B를 추가하여 학습합니다. 전체 파라미터의 약 1% 미만만 학습하면서도 풀 파인튜닝에 근접한 성능을 달성할 수 있습니다. 핵심 수식은 W' = W + BA이며, r(랭크)값이 작을수록 학습 파라미터가 줄어듭니다. RTX 4060(8GB) 같은 소비자용 GPU에서도 7B 모델을 파인튜닝할 수 있다는 것이 최대 장점입니다."
    }
  ]
}

✅ ChatML 형식으로 저장: /home/hpe/LLM_master_5parts/data/samples/chatml_ko_sample.json


---
## 6️⃣ 7. HuggingFace datasets 라이브러리 활용

In [9]:
# HuggingFace datasets 라이브러리 사용법
print("📚 HuggingFace datasets 라이브러리 활용")
print("=" * 60)

try:
    from datasets import Dataset, DatasetDict, load_dataset
    print("✅ datasets 라이브러리 import 성공\n")
    
    # 1. JSON 파일에서 Dataset 생성
    print("1️⃣ 로컬 JSON 파일에서 Dataset 생성")
    print("-" * 40)
    
    # Alpaca 데이터를 Dataset으로 변환
    dataset = Dataset.from_list(data)
    print(f"   Dataset: {dataset}")
    print(f"   컬럼: {dataset.column_names}")
    print(f"   첫 번째 데이터: {dataset[0]['instruction'][:50]}...")
    
    # 2. Train/Test 분할
    print(f"\n2️⃣ Train/Test 분할")
    print("-" * 40)
    
    split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
    print(f"   Train: {len(split_dataset['train'])}개")
    print(f"   Test:  {len(split_dataset['test'])}개")
    
    # 3. 데이터 매핑 (전처리)
    print(f"\n3️⃣ 데이터 매핑 (형식 변환)")
    print("-" * 40)
    
    def format_to_prompt(example):
        """Alpaca 형식을 프롬프트 문자열로 변환"""
        prompt = f"### 지시:\n{example['instruction']}\n\n"
        if example.get('input', '').strip():
            prompt += f"### 입력:\n{example['input']}\n\n"
        prompt += f"### 응답:\n{example['output']}"
        return {"text": prompt, "length": len(prompt)}
    
    formatted_dataset = dataset.map(format_to_prompt)
    print(f"   새 컬럼 추가: {formatted_dataset.column_names}")
    print(f"\n   📋 변환 결과 예시:")
    print(f"   {formatted_dataset[0]['text'][:200]}...")
    
    # 4. 필터링
    print(f"\n4️⃣ 데이터 필터링")
    print("-" * 40)
    
    long_data = formatted_dataset.filter(lambda x: x['length'] > 200)
    print(f"   200자 이상 데이터: {len(long_data)}개 / {len(formatted_dataset)}개")

except ImportError:
    print("⚠️ datasets 라이브러리가 설치되어 있지 않습니다.")
    print("   설치: pip install datasets")
    print("\n📝 코드 예시만 확인해주세요 (설치 후 실행 가능)")

📚 HuggingFace datasets 라이브러리 활용


/home/hpe/LLM_master_5parts/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ datasets 라이브러리 import 성공

1️⃣ 로컬 JSON 파일에서 Dataset 생성
----------------------------------------
   Dataset: Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 50
})
   컬럼: ['instruction', 'input', 'output']
   첫 번째 데이터: LoRA 파인튜닝이 무엇인지 설명하세요....

2️⃣ Train/Test 분할
----------------------------------------
   Train: 40개
   Test:  10개

3️⃣ 데이터 매핑 (형식 변환)
----------------------------------------


Map: 100%|██████████| 50/50 [00:00<00:00, 10962.63 examples/s]


   새 컬럼 추가: ['instruction', 'input', 'output', 'text', 'length']

   📋 변환 결과 예시:
   ### 지시:
LoRA 파인튜닝이 무엇인지 설명하세요.

### 응답:
LoRA(Low-Rank Adaptation)는 대규모 언어 모델을 효율적으로 파인튜닝하기 위한 기법입니다. 기존 모델의 가중치를 동결(freeze)하고, 각 레이어에 작은 저랭크(low-rank) 행렬 A와 B를 추가하여 학습합니다. 전체 파라미터의 약 1% 미만만 학습하면서도 풀 파...

4️⃣ 데이터 필터링
----------------------------------------


Filter: 100%|██████████| 50/50 [00:00<00:00, 25807.93 examples/s]

   200자 이상 데이터: 45개 / 50개


In [10]:
# HuggingFace Hub에서 데이터 다운로드 예시
print("📥 HuggingFace Hub 데이터 다운로드 예시")
print("=" * 60)

print("""
📌 유용한 한국어 데이터셋 (HuggingFace Hub):

# 1. KoAlpaca (범용 Instruction)
dataset = load_dataset("beomi/KoAlpaca-v1.1a")

# 2. NSMC (감성분석)
dataset = load_dataset("nsmc")

# 3. KorQuAD (질의응답)
dataset = load_dataset("squad_kor_v1")

# 4. Open-Orca Korean
dataset = load_dataset("kyujinpy/Open-platypus-Commercial")

# 5. 특정 split만 로드
train_data = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 6. streaming 모드 (대용량 데이터)
dataset = load_dataset("beomi/KoAlpaca-v1.1a", streaming=True)
for example in dataset['train'].take(5):
    print(example)
""")

print("💡 수강생 주의사항:")
print("  - 25명이 동시에 HuggingFace에서 다운로드하면 느려질 수 있습니다")
print("  - 미리 다운받은 데이터를 NAS/USB로 배포받으세요")
print("  - HF_HOME 환경변수로 캐시 경로를 공유 폴더로 설정 가능")

📥 HuggingFace Hub 데이터 다운로드 예시

📌 유용한 한국어 데이터셋 (HuggingFace Hub):

# 1. KoAlpaca (범용 Instruction)
dataset = load_dataset("beomi/KoAlpaca-v1.1a")

# 2. NSMC (감성분석)
dataset = load_dataset("nsmc")

# 3. KorQuAD (질의응답)
dataset = load_dataset("squad_kor_v1")

# 4. Open-Orca Korean
dataset = load_dataset("kyujinpy/Open-platypus-Commercial")

# 5. 특정 split만 로드
train_data = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 6. streaming 모드 (대용량 데이터)
dataset = load_dataset("beomi/KoAlpaca-v1.1a", streaming=True)
for example in dataset['train'].take(5):
    print(example)

💡 수강생 주의사항:
  - 25명이 동시에 HuggingFace에서 다운로드하면 느려질 수 있습니다
  - 미리 다운받은 데이터를 NAS/USB로 배포받으세요
  - HF_HOME 환경변수로 캐시 경로를 공유 폴더로 설정 가능


---
## 7️⃣ 8. 데이터 품질 검증 체크리스트

파인튜닝 전 반드시 확인해야 할 데이터 품질 항목들입니다.

In [11]:
# 데이터 품질 검증 도구
print("✅ 데이터 품질 검증 체크리스트")
print("=" * 60)

def validate_dataset(data, format_type="alpaca"):
    """데이터셋 품질 검증"""
    checks = []
    
    # 1. 데이터 수 확인
    n = len(data)
    checks.append({
        "check": "데이터 수",
        "result": f"{n}개",
        "status": "✅" if n >= 100 else "⚠️" if n >= 10 else "❌",
        "note": "최소 100개 이상 권장, 1000개 이상이 이상적"
    })
    
    # 2. 형식 검증
    if format_type == "alpaca":
        valid_format = sum(1 for d in data if "instruction" in d and "output" in d)
    else:
        valid_format = sum(1 for d in data if "messages" in d)
    
    checks.append({
        "check": "형식 일관성",
        "result": f"{valid_format}/{n} ({valid_format/n*100:.0f}%)",
        "status": "✅" if valid_format == n else "❌",
        "note": "100% 필수"
    })
    
    # 3. 중복 확인
    unique_instructions = len(set(d.get("instruction", "") for d in data))
    dup_ratio = (n - unique_instructions) / n * 100
    checks.append({
        "check": "중복 비율",
        "result": f"{dup_ratio:.1f}%",
        "status": "✅" if dup_ratio < 5 else "⚠️" if dup_ratio < 20 else "❌",
        "note": "5% 미만 권장"
    })
    
    # 4. 빈 필드 확인
    empty_output = sum(1 for d in data if not d.get("output", "").strip())
    checks.append({
        "check": "빈 응답",
        "result": f"{empty_output}개",
        "status": "✅" if empty_output == 0 else "❌",
        "note": "0개 필수"
    })
    
    # 5. 길이 분포
    output_lens = [len(d.get("output", "")) for d in data]
    avg_len = np.mean(output_lens)
    checks.append({
        "check": "평균 응답 길이",
        "result": f"{avg_len:.0f}자",
        "status": "✅" if 50 < avg_len < 1000 else "⚠️",
        "note": "50~1000자 권장"
    })
    
    # 6. 극단값 확인
    too_short = sum(1 for l in output_lens if l < 10)
    too_long = sum(1 for l in output_lens if l > 2000)
    checks.append({
        "check": "극단값 (너무 짧음)",
        "result": f"{too_short}개 (<10자)",
        "status": "✅" if too_short == 0 else "⚠️",
        "note": "제거 권장"
    })
    checks.append({
        "check": "극단값 (너무 김)",
        "result": f"{too_long}개 (>2000자)",
        "status": "✅" if too_long == 0 else "⚠️",
        "note": "max_seq_length 초과 시 잘림"
    })
    
    # 리포트 출력
    print(f"\n{'체크 항목':>20s} | {'결과':>20s} | {'상태':>4s} | {'기준/참고'}")
    print("-" * 80)
    for c in checks:
        print(f"{c['check']:>20s} | {c['result']:>20s} | {c['status']:>4s} | {c['note']}")
    
    passed = sum(1 for c in checks if c['status'] == '✅')
    total = len(checks)
    print(f"\n📊 검증 결과: {passed}/{total} 통과")
    
    return checks

# 검증 실행
validate_dataset(data, format_type="alpaca")

✅ 데이터 품질 검증 체크리스트

               체크 항목 |                   결과 |   상태 | 기준/참고
--------------------------------------------------------------------------------
               데이터 수 |                  50개 |   ⚠️ | 최소 100개 이상 권장, 1000개 이상이 이상적
              형식 일관성 |         50/50 (100%) |    ✅ | 100% 필수
               중복 비율 |                 8.0% |   ⚠️ | 5% 미만 권장
                빈 응답 |                   0개 |    ✅ | 0개 필수
            평균 응답 길이 |                 223자 |    ✅ | 50~1000자 권장
         극단값 (너무 짧음) |            0개 (<10자) |    ✅ | 제거 권장
          극단값 (너무 김) |          0개 (>2000자) |    ✅ | max_seq_length 초과 시 잘림

📊 검증 결과: 5/7 통과


[{'check': '데이터 수',
  'result': '50개',
  'status': '⚠️',
  'note': '최소 100개 이상 권장, 1000개 이상이 이상적'},
 {'check': '형식 일관성',
  'result': '50/50 (100%)',
  'status': '✅',
  'note': '100% 필수'},
 {'check': '중복 비율', 'result': '8.0%', 'status': '⚠️', 'note': '5% 미만 권장'},
 {'check': '빈 응답', 'result': '0개', 'status': '✅', 'note': '0개 필수'},
 {'check': '평균 응답 길이', 'result': '223자', 'status': '✅', 'note': '50~1000자 권장'},
 {'check': '극단값 (너무 짧음)',
  'result': '0개 (<10자)',
  'status': '✅',
  'note': '제거 권장'},
 {'check': '극단값 (너무 김)',
  'result': '0개 (>2000자)',
  'status': '✅',
  'note': 'max_seq_length 초과 시 잘림'}]

---

# 🧪 Part B — 합성 데이터 & Distillation

이 영역은 Part A에서 다룬 데이터 형식·정제 기법을 바탕으로, **LLM API를 활용해 학습 데이터를 자동 생성**하고 **큰 모델의 지식을 작은 모델에 증류(distillation)** 하는 방법을 다룹니다.

> ⚠️ 아래 셀들 일부는 OpenAI API 호출을 포함합니다. 크레딧이 충분한지 확인하세요.

---

---
## 🎯 1. 합성 데이터와 Distillation 개요

### 합성 데이터(Synthetic Data)란?

실제 데이터가 아닌, **AI 모델이 생성한 인공 데이터**를 의미합니다.

```
🧠 Teacher Model (GPT-4)
    ↓ 데이터 생성
📚 합성 학습 데이터 (수천~수만개)
    ↓ 학습
🤖 Student Model (Qwen2.5-1.5B)
```

### Knowledge Distillation이란?

큰 모델(Teacher)의 **지식을 작은 모델(Student)**에게 전달하는 기법

| 항목 | Teacher | Student |
|------|---------|----------|
| 모델 | GPT-4, Claude | Qwen2.5-1.5B |
| 파라미터 | 수천억 | 15억 |
| 비용 | API 비용 | 로컬 무료 |
| 속도 | 느림 | 빠름 |
| 프라이버시 | 외부 서버 | 로컬 환경 |

---
## 1️⃣ 2. 합성 데이터 생성이 필요한 이유

### 현실적인 문제들

| 문제 | 설명 | 합성 데이터 해결 |
|------|------|----------------|
| 📌 **데이터 부족** | 도메인 특화 데이터가 적음 | LLM으로 대량 생성 |
| 📌 **비용 문제** | 전문가 라벨링 비용 높음 | API 비용으로 대체 |
| 📌 **프라이버시** | 실제 데이터 활용 제한 | 합성 데이터로 우회 |
| 📌 **다양성 부족** | 편향된 데이터 | 다양한 시나리오 생성 |
| 📌 **시간 문제** | 데이터 수집에 수개월 | 수시간~수일 |

### 합성 데이터의 성공 사례

- 🏆 **Alpaca (Stanford)**: GPT-3.5로 52K 데이터 생성 → 7B 모델 파인튜닝
- 🏆 **Vicuna**: ShareGPT 대화 데이터 → 13B 모델이 GPT-4의 90% 성능
- 🏆 **Orca**: GPT-4의 추론 과정 학습 → 작은 모델의 추론 능력 향상
- 🏆 **phi-1.5 (Microsoft)**: 합성 교과서 데이터로 1.3B 모델이 5~25B 모델 성능

---
## 2️⃣ 3. Self-Instruct 방법론

### Self-Instruct 프로세스

2022년 Wang et al.이 제안한 방법으로, LLM이 스스로 학습 데이터를 생성하는 부트스트래핑 기법입니다.

```
📝 Seed Instructions (초기 시드 데이터 ~175개)
    ↓
🤖 LLM이 새로운 Instruction 생성
    ↓
🔍 품질 필터링 (중복 제거, 유사도 체크)
    ↓
🤖 LLM이 각 Instruction에 대한 Output 생성
    ↓
📚 새로운 학습 데이터 (52K+)
    ↓
🔄 반복 (새 데이터를 시드에 추가)
```

### 핵심 원리

1. **다양성 확보**: 시드 데이터의 카테고리를 다양하게
2. **품질 필터링**: 생성된 데이터 중 고품질만 선별
3. **반복 확장**: 좋은 데이터를 시드에 추가하여 재귀적 확장

In [12]:
# 📝 Seed 데이터 로드
print("📝 Seed 데이터 로드")
print("=" * 60)

# data/samples/alpaca_ko_sample.json 에서 Seed 로드
seed_file = os.path.join(DATA_DIR, "alpaca_ko_sample.json")

with open(seed_file, 'r', encoding='utf-8') as f:
    full_data = json.load(f)

print(f"📂 로드된 데이터: {seed_file}")
print(f"   총 {len(full_data)}건")

# Seed로 사용할 5개만 선택
seed_data = full_data[:5]

print(f"\n🌱 Seed 데이터 ({len(seed_data)}개):")
for i, item in enumerate(seed_data, 1):
    print(f"  {i}. {item['instruction'][:60]}")
    if item.get('input'):
        print(f"     입력: {item['input'][:40]}")
    print(f"     출력: {item['output'][:60]}...")

📝 Seed 데이터 로드
📂 로드된 데이터: /home/hpe/LLM_master_5parts/data/samples/alpaca_ko_sample.json
   총 50건

🌱 Seed 데이터 (5개):
  1. LoRA 파인튜닝이 무엇인지 설명하세요.
     출력: LoRA(Low-Rank Adaptation)는 대규모 언어 모델을 효율적으로 파인튜닝하기 위한 기법입니다....
  2. LoRA에서 rank(r)를 높이면 어떤 효과가 있나요?
     출력: LoRA에서 rank(r)를 높이면 학습 가능한 파라미터 수가 증가합니다. r=8이면 적은 패턴만 학습하고,...
  3. QLoRA가 일반 LoRA와 어떻게 다른가요?
     출력: QLoRA는 LoRA에 양자화(Quantization)를 결합한 기법입니다. 기본 모델을 NF4(4비트)로 ...
  4. 파인튜닝과 프롬프트 엔지니어링의 차이점을 설명하세요.
     출력: 파인튜닝은 모델의 가중치를 직접 수정하여 특정 태스크에 최적화하는 방법입니다. 학습 데이터와 GPU가 필요하...
  5. SFT(Supervised Fine-Tuning)란 무엇인가요?
     출력: SFT는 지도 학습 기반 파인튜닝으로, 사람이 작성한 고품질 instruction-response 쌍으로 모...


---
## 3️⃣ 4. GPT-4를 활용한 데이터 생성 (OpenAI API)

In [13]:
# 🔧 데이터 생성 프롬프트 + API 호출 함수
print("🔧 데이터 생성 함수 정의")
print("=" * 60)

SYSTEM_PROMPT = """당신은 AI 학습 데이터 생성 전문가입니다.
주어진 시드 데이터를 참고하여, 새로운 instruction-output 쌍을 생성합니다.

규칙:
1. 시드 데이터와 유사하지만 다른 새로운 질문을 만드세요
2. 한국어로 작성하세요
3. output은 구체적이고 정확해야 합니다
4. 반드시 JSON 배열 형식으로만 출력하세요 (다른 텍스트 없이)"""


def generate_data_batch(client, seed_examples, n_generate=5, category=None):
    """GPT-4o-mini로 학습 데이터 배치 생성"""
    # Seed 예시 → 프롬프트 구성
    examples = random.sample(seed_examples, min(3, len(seed_examples)))
    example_text = json.dumps([
        {"instruction": e["instruction"], "input": e.get("input", ""), "output": e["output"]}
        for e in examples
    ], ensure_ascii=False, indent=2)
    
    category_hint = f"\n카테고리: {category}" if category else ""
    
    prompt = f"""다음은 기존 학습 데이터 예시입니다:

{example_text}

위 예시를 참고하여, 새로운 {n_generate}개의 instruction-output 쌍을 생성하세요.{category_hint}
반드시 JSON 배열 형식으로만 출력하세요.
[{{"instruction": "...", "input": "...", "output": "..."}}, ...]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.8,
        max_tokens=2000,
    )
    
    content = response.choices[0].message.content.strip()
    # 코드 블록 제거
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("json"):
            content = content[4:]
    
    generated = json.loads(content)
    usage = response.usage
    
    return generated, {
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }

print("✅ generate_data_batch() 정의 완료")

🔧 데이터 생성 함수 정의
✅ generate_data_batch() 정의 완료


In [14]:
# 🚀 테스트: 1개 배치 생성해보기
print("🚀 테스트: Seed 5개 → 새 데이터 5개 생성")
print("=" * 60)

generated, usage = generate_data_batch(client, seed_data, n_generate=5)

print(f"✅ {len(generated)}개 생성 완료!")
print(f"   토큰 사용: {usage['total_tokens']:,}")
print()

for i, item in enumerate(generated, 1):
    print(f"  {i}. {item['instruction'][:70]}")
    if item.get('input'):
        print(f"     입력: {item['input'][:50]}")
    print(f"     출력: {item['output'][:80]}...")
    print()

🚀 테스트: Seed 5개 → 새 데이터 5개 생성
✅ 5개 생성 완료!
   토큰 사용: 1,328

  1. LoRA의 장점은 무엇인가요?
     출력: LoRA의 주요 장점은 메모리 사용량을 크게 줄이면서도 효율적으로 모델을 파인튜닝할 수 있다는 점입니다. 전체 파라미터의 소량만을 학습하기 때문...

  2. SFT의 단점은 무엇인가요?
     출력: SFT의 단점은 고품질 데이터셋이 필요하다는 것입니다. 사람이 작성한 instruction-response 쌍이 부족하면 모델의 성능이 저하될 ...

  3. LoRA를 사용하여 모델을 개선하기 위한 절차를 설명해 주세요.
     출력: LoRA를 사용하여 모델을 개선하기 위한 절차는 다음과 같습니다. 첫째, 기존의 사전학습된 모델을 로드합니다. 둘째, 각 레이어에 저랭크 행렬 ...

  4. QLoRA의 활용 사례는 무엇인가요?
     출력: QLoRA는 자원이 제한된 환경에서 대규모 언어 모델을 파인튜닝할 때 유용하게 사용됩니다. 예를 들어, 연구자들이 7B 파라미터 모델을 8GB ...

  5. LoRA와 QLoRA 각각의 경우, 어떤 상황에서 선택해야 할까요?
     출력: LoRA는 GPU 메모리 여유가 있는 상황에서, 즉 대규모 모델을 효과적으로 파인튜닝하고자 할 때 적합합니다. 반면, QLoRA는 메모리가 제한...



---
## 4️⃣ 5. 실습: Seed 기반 대규모 확장 (5개 → 30개)

카테고리별로 데이터를 자동 생성하는 파이프라인을 실행합니다.

In [15]:
# 🚀 카테고리별 데이터 생성 파이프라인
print("🚀 카테고리별 데이터 생성 파이프라인")
print("=" * 60)

categories = ["번역", "코딩", "요약"]  # 3개 카테고리 × 10개 = 30개

all_generated = []
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

print(f"📌 {len(categories)}개 카테고리 × 10개 = 총 {len(categories)*10}개 생성")
print(f"📌 모델: gpt-4o-mini")
print()

for i, category in enumerate(categories):
    print(f"  [{i+1}/{len(categories)}] {category}...", end=" ")
    
    generated, usage = generate_data_batch(
        client, seed_data, n_generate=10, category=category
    )
    
    for item in generated:
        item["category"] = category
    all_generated.extend(generated)
    
    for key in total_usage:
        total_usage[key] += usage.get(key, 0)
    
    print(f"✅ {len(generated)}개")
    time.sleep(1)

# 비용 계산
input_cost = total_usage['prompt_tokens'] / 1_000_000 * 0.15
output_cost = total_usage['completion_tokens'] / 1_000_000 * 0.60

print(f"\n{'='*60}")
print(f"📊 결과: 총 {len(all_generated)}개 생성")
print(f"   토큰: {total_usage['total_tokens']:,}")
print(f"   비용: ${input_cost + output_cost:.4f}")

🚀 카테고리별 데이터 생성 파이프라인
📌 3개 카테고리 × 10개 = 총 30개 생성
📌 모델: gpt-4o-mini

  [1/3] 번역... ✅ 10개


NameError: name 'time' is not defined

In [ ]:
# 생성 데이터 확인
print("📋 생성된 데이터 샘플 확인")
print("=" * 60)

if all_generated:
    # 카테고리별 분포
    from collections import Counter
    cat_counter = Counter(item.get('category', '미분류') for item in all_generated)
    
    print("📊 카테고리별 분포:")
    for cat, count in sorted(cat_counter.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"  {cat:>10s}: {count:>3d} {bar}")
    
    # 랜덤 샘플 표시
    print(f"\n📋 랜덤 샘플 3개:")
    samples = random.sample(all_generated, min(3, len(all_generated)))
    for i, sample in enumerate(samples, 1):
        print(f"\n  {'─'*50}")
        print(f"  📌 샘플 {i} [{sample.get('category', '미분류')}]")
        print(f"  Instruction: {sample['instruction'][:80]}")
        if sample.get('input', '').strip():
            print(f"  Input: {sample['input'][:80]}")
        print(f"  Output: {sample['output'][:100]}...")
else:
    print("⚠️ 생성된 데이터가 없습니다.")

---
## 5️⃣ 6. Distillation: 큰 모델 → 작은 모델 지식 전달

### Distillation의 두 가지 방식

#### 방식 1: 데이터 Distillation (우리가 할 것)
```
GPT-4 (Teacher) → 고품질 응답 생성 → 학습 데이터
                                         ↓
Qwen-1.5B (Student) ← SFT 학습 ← 학습 데이터
```

#### 방식 2: Logit Distillation (전통적 방식)
```
Teacher Model → Soft Labels (확률 분포)
                     ↓
Student Model → 확률 분포를 학습 (KL Divergence)
```

### 데이터 Distillation 전략

| 전략 | 설명 | 효과 |
|------|------|------|
| **Simple Distillation** | GPT-4의 응답을 그대로 학습 | 기본 |
| **Chain-of-Thought** | 추론 과정까지 포함하여 생성 | 추론 능력 향상 |
| **Step-by-Step** | 단계별 설명을 포함 | 설명 능력 향상 |
| **Critique + Revision** | 자가 비평 + 수정 과정 포함 | 자기교정 능력 |

In [ ]:
# 🔧 Distillation 3가지 전략 비교 (실제 API 호출)
print("🔧 Distillation: 동일 질문 → 3가지 전략으로 응답 생성")
print("=" * 60)

question = "트랜스포머 모델의 어텐션 메커니즘을 설명해주세요."
print(f"❓ 질문: {question}\n")

strategies = {
    "Simple": "당신은 유능한 AI 어시스턴트입니다. 정확하고 유용한 답변을 제공하세요.",
    "CoT": """당신은 전문 교육자입니다.
질문에 답할 때 먼저 생각 과정을 [생각 과정] 섹션에 보여주고, 그 다음 [답변] 섹션에 최종 답변을 제공하세요.""",
    "Step-by-Step": """당신은 전문 교육자입니다.
질문에 답할 때 단계별로 상세하게 설명하세요. 각 단계는 '1단계:', '2단계:' 형식으로 구분하세요.""",
}

distill_comparison = {}

for name, system_prompt in strategies.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0.7,
        max_tokens=500,
    )
    answer = response.choices[0].message.content.strip()
    distill_comparison[name] = answer
    
    print(f"📋 전략: {name}")
    print(f"{'─'*50}")
    print(f"{answer[:300]}...")
    print()

print("💡 CoT/Step-by-Step 전략은 작은 모델의 추론 능력을 향상시킵니다.")
print("   Orca 논문에서 이 접근법의 효과가 입증되었습니다.")

In [ ]:
# 🚀 Distillation 데이터 대량 생성
print("🚀 CoT 전략으로 Distillation 데이터 생성")
print("=" * 60)

distill_questions = [
    "파이썬에서 데코레이터는 어떻게 동작하나요?",
    "GPT와 BERT의 차이점은 무엇인가요?",
    "LoRA 파인튜닝이 Full Fine-Tuning보다 효율적인 이유는?",
    "RAG 시스템에서 벡터 데이터베이스의 역할은?",
    "양자화(Quantization)가 모델 성능에 미치는 영향은?",
]

cot_system = """당신은 전문 교육자입니다.
질문에 답할 때 먼저 생각 과정을 보여주고, 그 다음 최종 답변을 제공하세요.
형식:
[생각 과정]
...
[답변]
..."""

distill_data = []
total_tokens = 0

for i, q in enumerate(distill_questions):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": cot_system},
            {"role": "user", "content": q}
        ],
        temperature=0.7,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()
    total_tokens += response.usage.total_tokens
    
    distill_data.append({
        "instruction": q,
        "input": "",
        "output": answer,
    })
    print(f"  [{i+1}/{len(distill_questions)}] ✅ {q[:40]}... ({len(answer)}자)")
    time.sleep(0.5)

print(f"\n📊 총 {len(distill_data)}개 생성, 토큰: {total_tokens:,}")
print(f"\n📋 샘플 (첫 번째):")
print(f"  Q: {distill_data[0]['instruction']}")
print(f"  A: {distill_data[0]['output'][:200]}...")

---
## 6️⃣ 7. 생성 데이터 품질 검증 및 필터링

In [ ]:
# 🔍 품질 검증 + 최종 데이터 저장
print("🔍 생성 데이터 품질 검증 및 저장")
print("=" * 60)

def quality_filter(data, min_output_len=20):
    """생성 데이터 품질 필터링"""
    filtered = []
    seen = set()
    removed = {"empty": 0, "short": 0, "duplicate": 0}
    
    for item in data:
        instruction = item.get("instruction", "").strip()
        output = item.get("output", "").strip()
        
        if not instruction or not output:
            removed["empty"] += 1
            continue
        if len(output) < min_output_len:
            removed["short"] += 1
            continue
        if instruction in seen:
            removed["duplicate"] += 1
            continue
        seen.add(instruction)
        filtered.append(item)
    
    return filtered, removed

# 합성 데이터 필터링
print("1️⃣ 합성 데이터 (Self-Instruct)")
filtered_synthetic, stats1 = quality_filter(all_generated)
print(f"   입력: {len(all_generated)}개 → 통과: {len(filtered_synthetic)}개")
print(f"   제거: 빈 데이터 {stats1['empty']}, 짧은 응답 {stats1['short']}, 중복 {stats1['duplicate']}")

# Distillation 데이터 필터링
print("\n2️⃣ Distillation 데이터 (CoT)")
filtered_distill, stats2 = quality_filter(distill_data)
print(f"   입력: {len(distill_data)}개 → 통과: {len(filtered_distill)}개")

# 통합 저장
final_data = seed_data + filtered_synthetic + filtered_distill

# instruction, input, output만 저장
save_data = [
    {k: v for k, v in item.items() if k in ["instruction", "input", "output"]}
    for item in final_data
]

output_file = os.path.join(DATA_DIR, "synthetic_data.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print(f"✅ 최종 저장: {output_file}")
print(f"   Seed: {len(seed_data)}개 + 합성: {len(filtered_synthetic)}개 + Distill: {len(filtered_distill)}개")
print(f"   합계: {len(save_data)}개")

---
## 7️⃣ 8. 비용 계산 및 최적화 팁

In [ ]:
# API 비용 계산기
print("💰 데이터 생성 비용 계산기")
print("=" * 60)

# 모델별 가격 (2024년 기준, 100만 토큰당)
pricing = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
}

def estimate_cost(n_samples, avg_input_tokens=500, avg_output_tokens=300, model="gpt-4o-mini"):
    """데이터 생성 비용 추정"""
    prices = pricing.get(model, pricing["gpt-4o-mini"])
    
    total_input = n_samples * avg_input_tokens
    total_output = n_samples * avg_output_tokens
    
    input_cost = total_input / 1_000_000 * prices["input"]
    output_cost = total_output / 1_000_000 * prices["output"]
    total_cost = input_cost + output_cost
    
    return {
        "model": model,
        "n_samples": n_samples,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

# 다양한 시나리오 비용 계산
scenarios = [
    (100, "gpt-4o-mini", "소규모 실습"),
    (1000, "gpt-4o-mini", "중규모 프로젝트"),
    (10000, "gpt-4o-mini", "대규모 프로젝트"),
    (1000, "gpt-4o", "고품질 데이터 (gpt-4o)"),
    (1000, "gpt-4-turbo", "최고 품질 (gpt-4-turbo)"),
]

print(f"\n{'시나리오':>25s} | {'데이터 수':>10s} | {'모델':>15s} | {'예상 비용':>12s}")
print("-" * 75)

for n, model, desc in scenarios:
    result = estimate_cost(n, model=model)
    print(f"{desc:>25s} | {n:>10,d} | {model:>15s} | ${result['total_cost']:>10.2f}")

print("\n💡 비용 최적화 팁:")
print("  1️⃣ gpt-4o-mini 사용 → gpt-4o 대비 약 10~20배 저렴")
print("  2️⃣ 배치 생성: 한 번에 여러 개 생성하면 프롬프트 토큰 절약")
print("  3️⃣ 프롬프트 최적화: 시드 예시 수를 최소화")
print("  4️⃣ 점진적 접근: 100개 → 검증 → 1000개 → 검증 → 확장")
print("  5️⃣ 캐싱: 동일 프롬프트 재사용 방지")

In [ ]:
# 비용 대비 효과 분석
print("📊 데이터 양 vs 성능 (일반적 트렌드)")
print("=" * 60)

print("""
📌 일반적으로 관찰되는 데이터 양 vs 파인튜닝 성능:

  데이터 수  | 예상 품질  | 비용 (4o-mini) | 권장 단계
  ─────────┼──────────┼──────────────┼──────────
    100개   | ★★☆☆☆   |    ~$0.05    | 1. 프로토타이핑
    500개   | ★★★☆☆   |    ~$0.25    | 2. 초기 실험
  1,000개   | ★★★★☆   |    ~$0.50    | 3. 기본 파인튜닝
  5,000개   | ★★★★☆   |    ~$2.50    | 4. 준 프로덕션
 10,000개   | ★★★★★   |    ~$5.00    | 5. 프로덕션
 50,000개   | ★★★★★   |   ~$25.00    | 6. 고성능 (수확체감)

💡 핵심 인사이트:
  - 1,000~5,000개에서 가성비가 가장 좋음
  - 10,000개 이상부터는 수확체감(diminishing returns)
  - 데이터 품질이 양보다 중요 (LIMA 논문)
  - 도메인이 좁을수록 적은 데이터로도 효과적
""")

---

## 📝 정리 및 핵심 요약

### Part A 요약 — 데이터 파이프라인

| 단계 | 핵심 내용 |
|------|----------|
| 형식 | Alpaca / ShareGPT / ChatML — 모델/프레임워크 별 통일된 형식 사용 |
| 수집 | 공개 데이터셋(HF Hub), 내부 데이터, 크롤링, 사용자 로그 |
| 정제 | 중복 제거, 길이 필터, 욕설/PII 제거, 응답 품질 검사 |
| 증강 | 템플릿 다양화, paraphrase, instruction back-translation |
| 변환 | `datasets.Dataset` 객체 ↔ JSONL ↔ Parquet 자유로운 변환 |
| 검증 | 길이 분포 / 라벨 균형 / 중복률 / 인코딩 오류 체크리스트 |

### Part B 요약 — 합성 데이터 & Distillation

| 항목 | 핵심 내용 |
|------|----------|
| 합성 데이터 필요성 | 도메인 특화 / 사이즈 부족 / 라벨 비용 절감 |
| Self-Instruct | 소수 seed → LLM이 수만 개로 확장 |
| OpenAI API 패턴 | system prompt + few-shot seed → JSON 응답 강제 |
| Distillation | Teacher(예: GPT-4o) 응답을 Student(예: Qwen2.5-1.5B) 학습 데이터로 |
| 품질 필터링 | 길이 / 중복 / format 검증 / LLM-as-Judge |
| 비용 최적화 | 배치 호출, gpt-4o-mini 활용, 캐싱, prompt 압축 |

### 다음 세션 예고

- 🔜 **Session 14**: Next Token Prediction 기반 SFT 실습 — 본 세션에서 준비한 데이터로 HuggingFace Trainer / TRL SFTTrainer 학습 실행

---

### 💡 실습 과제

1. 본인 관심 도메인의 seed 데이터 5개를 직접 작성 → Part B 코드로 30개로 확장
2. 생성된 데이터를 Alpaca → ChatML로 변환 (Part A 코드 활용)
3. 품질 검증 체크리스트 적용 후 `datasets`로 저장
4. (선택) gpt-4o vs gpt-4o-mini로 동일 seed 생성 비교 — 비용 대비 품질 차이 측정

---

In [ ]:
print("✅ Session 13 완료!")
print("📚 다음 세션: Session 14 — SFT 실습 (HuggingFace Trainer / TRL SFTTrainer)")